In [ ]:
import pandas as pd
from rdkit import Chem
import torch

from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
import numpy as np

from sklearn.model_selection import train_test_split
from itertools import product
import random


import sys
import os
sys.path.append(os.path.abspath('../'))
from reduceGraph import mol_to_graph, graph_to_pyg_oh, reduce_graph_from_mol_oh, mol_to_pool_idx
from networks import GAT, PPGAT

In [4]:
#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [5]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.7.0+cu126
12.6
True


In [6]:
#DATASET 

dataset = pd.read_csv('BBBP.csv')
dataset


,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...
...,...,...,...,...
2045,2049,licostinel,1,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl
2046,2050,ademetionine(adenosyl-methionine),1,[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...
2047,2051,mesocarb,1,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...
2048,2052,tofisoline,1,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...


In [7]:
dataset.p_np.value_counts()

p_np
1    1567
0     483
Name: count, dtype: int64

In [8]:
def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    graph = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(graph)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_pyg_dataset(df, smiles_col, label_col):
    data_list = []
    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [9]:
bbbp_dataset = dataframe_to_pyg_dataset(dataset, 'smiles', 'p_np')


[12:06:40] Explicit valence for atom # 1 N, 4, is greater than permitted
[12:06:40] WARNING: not removing hydrogen atom without neighbors
[12:06:40] Explicit valence for atom # 6 N, 4, is greater than permitted
[12:06:40] WARNING: not removing hydrogen atom without neighbors
[12:06:40] WARNING: not removing hydrogen atom without neighbors
[12:06:40] WARNING: not removing hydrogen atom without neighbors
[12:06:41] WARNING: not removing hydrogen atom without neighbors
[12:06:41] WARNING: not removing hydrogen atom without neighbors
[12:06:41] WARNING: not removing hydrogen atom without neighbors
[12:06:41] Explicit valence for atom # 6 N, 4, is greater than permitted
[12:06:41] WARNING: not removing hydrogen atom without neighbors
[12:06:41] WARNING: not removing hydrogen atom without neighbors
[12:06:41] WARNING: not removing hydrogen atom without neighbors
[12:06:42] WARNING: not removing hydrogen atom without neighbors
[12:06:42] Explicit valence for atom # 11 N, 4, is greater than pe

In [10]:
#grid search
learning_rates = [1e-4, 5e-4, 1e-3]
batch_sizes = [16, 32, 64]
grid = list(product(learning_rates, batch_sizes))

In [11]:
#multi label aware stratified split


def train_eval_model(mod, dataset, in_channels, edge_attr_dim, out_channels, grid, epochs=30):
    labels = [int(data.y.item()) for data in dataset]

    train_val_indices, test_indices = train_test_split(
        list(range(len(dataset))),
        test_size=0.2,
        stratify=labels,
        random_state=42
    )

    train_val_labels = [labels[i] for i in train_val_indices]

    # 10% of full = 12.5% of 80% → 0.125
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.125,
        stratify=train_val_labels,
        random_state=42
    )

    # Create subsets 
    train = [dataset[i] for i in train_indices]
    val = [dataset[i] for i in val_indices]
    test  = [dataset[i] for i in test_indices]


    #  Hyperparameter tuning 
    best_val_loss = float('inf')
    best_config = None
    best_model_state = None

    for lr, batch_size in grid:
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=batch_size)

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)
                loss = criterion(out, batch.y.float().unsqueeze(1))
                loss.backward()
                optimizer.step()

        # Evaluate on val
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to('cuda')
                out = model(batch)
                val_loss += criterion(out, batch.y.float().unsqueeze(1)).item() * batch.num_graphs


        val_loss /= len(val)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_config = (lr, batch_size)
            best_model_state = model.state_dict()

    #  Retrain final model on train + val 
    final_model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
    final_model.load_state_dict(best_model_state)

    full_train = train + val
    train_loader = DataLoader(full_train, batch_size=best_config[1], shuffle=True)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config[0])
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        final_model.train()
        for batch in train_loader:
            batch = batch.to('cuda')
            optimizer.zero_grad()
            out = final_model(batch)
            loss = criterion(out, batch.y.float().unsqueeze(1))
            loss.backward()
            optimizer.step()

    return final_model, best_config, best_val_loss

In [12]:
in_channels = bbbp_dataset[0].x.size(-1)
edge_attr_dim = bbbp_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(GAT, bbbp_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )


(0.001, 16)


In [14]:
def cross_validate_stratified(
    mod,
    dataset, in_channels, edge_attr_dim, out_channels,
    best_lr, best_batch_size, epochs=30, k=5
):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score
    import numpy as np
    import torch

    # Extract single target per sample
    y_vector = torch.stack([data.y.view(-1)[0] for data in dataset]).numpy()

    splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    all_aurocs = []
    all_bal_accs = []

    for fold, (train_val_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y_vector)), y_vector)):
        print(f"\n Fold {fold + 1}")

        # Split data
        y_train_val = y_vector[train_val_idx]
        train_val_data = [dataset[i] for i in train_val_idx]
        test_data = [dataset[i] for i in test_idx]

        # Inner val split
        inner_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=fold)
        train_idx, val_idx = next(inner_split.split(np.zeros(len(y_train_val)), y_train_val))
        train = [train_val_data[i] for i in train_idx]
        val = [train_val_data[i] for i in val_idx]

        # Initialize model
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=best_batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=best_batch_size)
        test_loader = DataLoader(test_data, batch_size=best_batch_size)

        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)

                y_true = batch.y.float()
                if y_true.dim() == 1:
                    y_true = y_true.unsqueeze(1)

                loss = criterion(out, y_true)
                loss.backward()
                optimizer.step()

        # Evaluate on test set
        model.eval()
        y_true, y_scores = [], []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to('cuda')
                out = model(batch)

                y_scores.append(out.cpu())
                y_true.append(batch.y.cpu())

        y_scores = torch.cat(y_scores).numpy()
        y_true = torch.cat(y_true).numpy()

        # AUROC
        auroc = roc_auc_score(y_true, y_scores)
        all_aurocs.append(auroc)

        # Balanced accuracy
        y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid
        y_pred = (y_probs >= 0.5).astype(int)

        bal_acc = balanced_accuracy_score(y_true, y_pred)
        all_bal_accs.append(bal_acc)

        print(f" Fold {fold + 1} AUROC: {auroc:.4f}")
        print(f" Fold {fold + 1} Balanced Acc: {bal_acc:.4f}")

    mean_auroc = np.mean(all_aurocs)
    std_auroc = np.std(all_aurocs)

    mean_bal_acc = np.mean(all_bal_accs)
    std_bal_acc = np.std(all_bal_accs)

    print(f"\n Mean AUROC = {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f" Mean Balanced Accuracy = {mean_bal_acc:.4f} ± {std_bal_acc:.4f}")

    return (
        all_aurocs, mean_auroc, std_auroc,
        all_bal_accs, mean_bal_acc, std_bal_acc
    )

In [16]:
results = cross_validate_stratified( GAT, 
    bbbp_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1


 Fold 1 AUROC: 0.8715
 Fold 1 Balanced Acc: 0.8065

 Fold 2
 Fold 2 AUROC: 0.8689
 Fold 2 Balanced Acc: 0.8089

 Fold 3
 Fold 3 AUROC: 0.8428
 Fold 3 Balanced Acc: 0.7340

 Fold 4
 Fold 4 AUROC: 0.8851
 Fold 4 Balanced Acc: 0.8089

 Fold 5
 Fold 5 AUROC: 0.8992
 Fold 5 Balanced Acc: 0.8320

 Mean AUROC = 0.8735 ± 0.0188
 Mean Balanced Accuracy = 0.7981 ± 0.0334


# RG

In [22]:

def smiles_to_rgdata(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    data = reduce_graph_from_mol_oh(mol)  # pytorch geometric RG graph from mol 
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_rg_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgdata(smiles, label)
        if data is not None:
            #add smiles to dataset 
            data.smiles = smiles            
            data_list.append(data)

    return data_list

In [23]:
bbbp_rg_dataset = dataframe_to_rg_pyg_dataset(dataset, 'smiles', 'p_np')

[13:18:11] Explicit valence for atom # 1 N, 4, is greater than permitted
[13:18:11] WARNING: not removing hydrogen atom without neighbors
[13:18:11] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:18:28] WARNING: not removing hydrogen atom without neighbors
[13:18:47] WARNING: not removing hydrogen atom without neighbors
[13:18:54] WARNING: not removing hydrogen atom without neighbors
[13:19:17] WARNING: not removing hydrogen atom without neighbors
[13:19:17] WARNING: not removing hydrogen atom without neighbors
[13:19:17] WARNING: not removing hydrogen atom without neighbors
[13:19:38] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:19:53] WARNING: not removing hydrogen atom without neighbors
[13:20:00] WARNING: not removing hydrogen atom without neighbors
[13:20:23] WARNING: not removing hydrogen atom without neighbors
[13:20:34] WARNING: not removing hydrogen atom without neighbors
[13:20:37] Explicit valence for atom # 11 N, 4, is greater than pe

In [24]:

model, config, loss = train_eval_model(GAT, bbbp_rg_dataset, bbbp_rg_dataset[0].x.size(-1), bbbp_rg_dataset[0].edge_attr.size(-1), 1, grid)

print(config )


(0.0005, 32)


In [25]:
result = cross_validate_stratified(
    GAT, 
    bbbp_rg_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8634
 Fold 1 Balanced Acc: 0.7640

 Fold 2
 Fold 2 AUROC: 0.8475
 Fold 2 Balanced Acc: 0.7704

 Fold 3
 Fold 3 AUROC: 0.8383
 Fold 3 Balanced Acc: 0.7640

 Fold 4
 Fold 4 AUROC: 0.8231
 Fold 4 Balanced Acc: 0.7079

 Fold 5
 Fold 5 AUROC: 0.8542
 Fold 5 Balanced Acc: 0.7798

 Mean AUROC = 0.8453 ± 0.0138
 Mean Balanced Accuracy = 0.7573 ± 0.0253


# PPGAT

In [26]:

def smiles_to_rgnn_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None  
    G = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(G)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 


    pharma_index, new_edge_index, new_edge_attr = mol_to_pool_idx(mol)
    data.pharma_index = pharma_index
    data.new_edge_index = new_edge_index
    data.new_edge_attr = new_edge_attr

 

    return data

def dataframe_to_rgnn_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgnn_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [27]:
bbbp_ppgat_dataset = dataframe_to_rgnn_pyg_dataset(dataset, 'smiles', 'p_np')

[13:57:58] Explicit valence for atom # 1 N, 4, is greater than permitted
[13:57:58] WARNING: not removing hydrogen atom without neighbors
[13:57:58] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:57:58] WARNING: not removing hydrogen atom without neighbors
[13:57:59] WARNING: not removing hydrogen atom without neighbors
[13:57:59] WARNING: not removing hydrogen atom without neighbors
[13:58:00] WARNING: not removing hydrogen atom without neighbors
[13:58:00] WARNING: not removing hydrogen atom without neighbors
[13:58:00] WARNING: not removing hydrogen atom without neighbors
[13:58:02] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:58:02] WARNING: not removing hydrogen atom without neighbors
[13:58:02] WARNING: not removing hydrogen atom without neighbors
[13:58:02] WARNING: not removing hydrogen atom without neighbors
[13:58:03] WARNING: not removing hydrogen atom without neighbors
[13:58:03] Explicit valence for atom # 11 N, 4, is greater than pe

In [28]:
model, config, loss = train_eval_model(PPGAT, bbbp_ppgat_dataset, bbbp_ppgat_dataset[0].x.size(-1), bbbp_ppgat_dataset[0].edge_attr.size(-1), 1, grid)
print(config)

(0.001, 64)


In [29]:
results = cross_validate_stratified(
    PPGAT, 
    bbbp_ppgat_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8499
 Fold 1 Balanced Acc: 0.7472

 Fold 2
 Fold 2 AUROC: 0.8542
 Fold 2 Balanced Acc: 0.7536

 Fold 3
 Fold 3 AUROC: 0.8707
 Fold 3 Balanced Acc: 0.7845

 Fold 4
 Fold 4 AUROC: 0.8908
 Fold 4 Balanced Acc: 0.7977

 Fold 5
 Fold 5 AUROC: 0.8847
 Fold 5 Balanced Acc: 0.8096

 Mean AUROC = 0.8701 ± 0.0161
 Mean Balanced Accuracy = 0.7785 ± 0.0244
